# Dependencias

In [76]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from functools import reduce

# Links y Carga de Datos

In [29]:
link = 'C:\\Users\\ramir\\Documents\\consultoria_coppel'

In [ ]:
df_seguro = pd.read_csv(link + '\\Clientes_CDPsalud.csv').drop(columns = ['Unnamed: 0'])
df_sociodemo = pd.read_csv( link + '\\sociodemograficos_clientes_salud.csv').drop(columns = ['Unnamed: 0'])

In [32]:
df_seguro['FechaMovimiento'].min()

'2024-01-01'

# Funciones

# Tratamieto de Datos

In [33]:
df_seguro['fechacorte_seguro'] = pd.to_datetime(df_seguro.fechacorte_seguro,
                                           format='%Y-%m-%d')
df_seguro['FechaMovimiento'] = pd.to_datetime(df_seguro.FechaMovimiento,
                                           format='%Y-%m-%d')
df_seguro['FechaVencimiento'] = pd.to_datetime(df_seguro.FechaVencimiento,
                                           format='%Y-%m-%d')
df_seguro['yearmonth_FechaMovimiento'] = df_seguro.FechaMovimiento.dt.to_period('M')

# Filtro de Tablas

In [34]:
# Podemos ver que existen clientes con más de un folio
df_cont_client_folio = df_seguro.groupby('ID_ficticio')[['Folio']].nunique()\
                                .reset_index().sort_values('Folio', ascending = False).rename(columns = {'Folio':'num_folios'})
df_seguro = df_seguro.merge(df_cont_client_folio.query('num_folios == 1')[['ID_ficticio']], how = 'inner')

In [35]:
df_seguro['FechaMovimiento'].min()

Timestamp('2024-01-01 00:00:00')

# Creación de Matriz de Covariables

In [72]:
ls_fchcorte = df_seguro['fechacorte_seguro'].unique()
# ls_fchcorte = [pd.to_datetime('2025-07-31')]

In [ ]:
ls_df = []
for fecha_analisis in ls_fchcorte:
    # nos traemos a todos los clientes vigentes al momento del análisis
    df_seg_activo = df_seguro[df_seguro['fechacorte_seguro']<= fecha_analisis]\
                                .reset_index(drop = True).copy()
    #  limitación de la tabla
    df_inicio_cliente = df_seg_activo[['ID_ficticio', 'FechaMovimiento']].groupby('ID_ficticio', as_index = False).min()
    df_inicio_cliente['antiguedad'] = (fecha_analisis - df_inicio_cliente['FechaMovimiento']).dt.days
    df_inicio_cliente = df_inicio_cliente[['ID_ficticio', 'antiguedad']]
    # # último pago 
    df_ult_pago = df_seg_activo[['ID_ficticio', 'FechaMovimiento']].groupby('ID_ficticio', as_index = False).max()
    df_ult_pago['ult_pago'] = (fecha_analisis - df_ult_pago['FechaMovimiento']).dt.days
    df_ult_pago = df_ult_pago[['ID_ficticio', 'ult_pago']]
    # # patron de pagos
    df_plazo_ult_pago = df_seg_activo[['ID_ficticio', 'yearmonth_FechaMovimiento', 'FechaMovimiento']]\
    .groupby(['ID_ficticio', 'yearmonth_FechaMovimiento'], as_index=False).count()\
    .sort_values(['ID_ficticio', 'yearmonth_FechaMovimiento'], ascending = False)\
    .drop_duplicates(subset = ['ID_ficticio'])[['ID_ficticio', 'FechaMovimiento']].reset_index(drop = True)\
    .rename(columns = {'FechaMovimiento':'plazo_ult_pago'})
    # # fecha de venicmiento
    df_futuro = df_seguro[df_seguro['fechacorte_seguro'] >= fecha_analisis]\
                                .reset_index(drop = True).copy()
    df_fecha_vencimiento = ((df_futuro.groupby('ID_ficticio')['FechaVencimiento'].min() -  fecha_analisis).dt.days)\
        .reset_index().rename(columns = {'FechaVencimiento':'dias_para_vencimiento'})
    
    df_final = df_fecha_vencimiento.merge(df_inicio_cliente).merge(df_ult_pago).merge(df_plazo_ult_pago)
    df_final['fechacorte_seguro'] = fecha_analisis
    ls_df.append(df_final)

In [91]:
df_covar = reduce(lambda left, right: pd.concat([left, right], axis = 0), ls_df).reset_index(drop = True)[['ID_ficticio',
                                                                                                           'fechacorte_seguro' ,
                                                                                                           'antiguedad',
                                                                                                             'ult_pago', 
                                                                                                             'plazo_ult_pago', 
                                                                                                             'dias_para_vencimiento']]

In [92]:
df_covar

,ID_ficticio,fechacorte_seguro,antiguedad,ult_pago,plazo_ult_pago,dias_para_vencimiento
0,8,2025-07-31,243,57,1,80
1,10,2025-07-31,225,225,12,171
2,17,2025-07-31,29,29,1,2
3,24,2025-07-31,469,16,1,18
4,29,2025-07-31,250,45,1,86
...,...,...,...,...,...,...
2578024,672102,2025-05-31,425,24,1,31
2578025,672110,2025-05-31,67,32,2,80
2578026,672112,2025-05-31,302,15,1,44
2578027,672114,2025-05-31,188,188,12,207


# Creación de Variable Target

In [ ]:
'''
Analizamos este cliente, lo que se propone para saber si el cliente va a dejar de pagar o no es el siguiente criterio 
Queremos ver si dada una fecha de corte, el cliente vuelve a pagar entre el periodo de la fecha de corte y tres meses después 
de la máxima fecha de vencimiento asociada a esa fecha corte: 
Ejemplo:
El siguiente cliente tiene como primera fecha de corte el 2024-05-31 (nos paramos en esa fecha en el tiempo)
y vemos que para esa fecha su máxima FechaVencimiento es 2025-05-20. Interpretamos que este cliente pagó 12 meses de seguro.
Para ver si perdimos la cliente o no, vemos si tiene una fecha de movimiento en el rengo

[fechacorte_seguro,  última FechaVencimiento + 3meses ] = [2024-05-31, 2025-08-20]

en caso de que sí: registramos como un 0 el que el cliente no abandonó el producto 
en caso de que no: registramos como un 1 el que el cliente abandonó el producto

En este caso, vemos que tiene una  fecha de movimiento el 2025-02-19 que está en el intervalo, por lo tanto, este cliente no abandonó el contrato.
Así nos seguimos para las otras fechas de corte e interpretamos cada nuevo registro como un cliente diferente (aunque sea el mismo) de esta 
manera aumentamos el número de registros en nuestra matriz de predictoras.

dado que nuestra última fecha de corte (en toda la table) es 2025-12-31 y no podemos saber si los clientes renovaron o no,
entonces no consideraremos las últimas 3 fechas de corte (2025-12-31, 2025-11-30, 2025-10-31). 
'''

# 1) Última fecha de vencimiento por cliente y fecha de corte
df_ult_fecha_venc = (
    df_seguro
    .sort_values(['ID_ficticio', 'fechacorte_seguro', 'FechaVencimiento'], ascending=[True, True, False])
    .drop_duplicates(['ID_ficticio', 'fechacorte_seguro'])
    [['ID_ficticio', 'fechacorte_seguro', 'FechaVencimiento']]
    .copy()
)

# 2) Máxima fecha de movimiento por cliente y fecha de corte
df_max_mov_x_fechacorte_id_fict = (
    df_seguro
    .sort_values(['ID_ficticio', 'fechacorte_seguro', 'FechaMovimiento'], ascending=[True, True, False])
    .drop_duplicates(['ID_ficticio', 'fechacorte_seguro'])
    [['ID_ficticio', 'fechacorte_seguro', 'FechaMovimiento']]
    .copy()
)

# 3) Ordenar cronológicamente por cliente para obtener la siguiente fecha de movimiento
df_max_mov_x_fechacorte_id_fict = (
    df_max_mov_x_fechacorte_id_fict
    .sort_values(['ID_ficticio', 'FechaMovimiento'])
    .copy()
)
df_max_mov_x_fechacorte_id_fict['FechaMovimiento_siguiente'] = (
    df_max_mov_x_fechacorte_id_fict
    .groupby('ID_ficticio')['FechaMovimiento']
    .shift(-1)
)

# 4) Dejar sólo columnas necesarias
df_max_mov_x_fechacorte_id_fict = df_max_mov_x_fechacorte_id_fict[
    ['ID_ficticio', 'fechacorte_seguro', 'FechaMovimiento_siguiente']
].copy()

# 5) Merge final
df_target = df_max_mov_x_fechacorte_id_fict.merge(
    df_ult_fecha_venc,
    on=['ID_ficticio', 'fechacorte_seguro'],
    how='inner'
)

# Definimos la variable Target 
df_target['abandono_prod'] = (
    df_target['FechaMovimiento_siguiente'].isna()
    |
    (
        df_target['FechaMovimiento_siguiente']
        > df_target['FechaVencimiento'] + pd.DateOffset(months=3)
    )
).astype(int)

# Quitamos los últimos 3 meses
df_target = df_target[~df_target['fechacorte_seguro'].isin(pd.to_datetime(['2025-12-31', '2025-11-30', '2025-10-31']))]

In [94]:
df_target = df_target[['ID_ficticio', 'fechacorte_seguro', 'abandono_prod']]

# Tabla Final

In [96]:
df_target.merge(df_covar, on = ['ID_ficticio', 'fechacorte_seguro'], how = 'left')

,ID_ficticio,fechacorte_seguro,abandono_prod,antiguedad,ult_pago,plazo_ult_pago,dias_para_vencimiento
0,1,2024-01-31,1,10,10,1,21
1,2,2024-01-31,1,29,29,1,2
2,3,2025-09-30,1,27,27,4,3
3,6,2024-01-31,1,4,4,1,29
4,8,2024-11-30,0,0,0,1,30
...,...,...,...,...,...,...,...
1454827,672116,2025-08-31,0,280,22,1,9
1454828,672116,2025-09-30,0,310,20,1,10
1454829,672117,2025-09-30,1,2,2,8,28
1454830,672118,2025-09-30,1,13,13,2,17


# Define additional covariables

In [ ]:
df_transac = pd.read_csv('transacciones_clientes_salud.csv')
df_transac.drop(columns=['Unnamed: 0'], inplace=True)

# Format date and create year_month variable
df_transac['fechacompra'] = pd.to_datetime(df_transac.fechacompra,
                                           format='%Y-%m-%d')
df_transac['year_month'] = df_transac.fechacompra.dt.to_period('M')

# Define transaction dataframe for sales only
df_transac_venta = (df_transac[(df_transac.precio_vta_perc.notnull()) &
                              (df_transac.precio_vta_perc > 0)]
                    .reset_index(drop=True)
                    .copy()
                    )
columns_to_keep = ['ID_ficticio', 'year_month', 'precio_vta_perc']
df_transac_venta = df_transac_venta[columns_to_keep]            
df_transac_venta.sort_values(['ID_ficticio', 'year_month'], inplace=True)

# Get sales behavior by customer, month and different time windows
df_transac_venta_1m = (df_transac_venta
                       .groupby(['ID_ficticio', 'year_month'])
                       .mean()
                       .reset_index()
                       .rename(columns={'precio_vta_perc': 'precio_vta_avg_1m'})
                       )
df_transac_venta_3m = (df_transac_venta_1m
                       .groupby(['ID_ficticio'])
                       .rolling(3, on='year_month', min_periods=1)
                       .mean()
                       .reset_index()
                       .drop(columns=['level_1'])
                       .rename(columns={'precio_vta_avg_1m': 'precio_vta_avg_3m'})
                       )
df_transac_venta_6m = (df_transac_venta_1m
                       .groupby(['ID_ficticio'])
                       .rolling(6, on='year_month', min_periods=1)
                       .mean()
                       .reset_index()
                       .drop(columns=['level_1'])
                       .rename(columns={'precio_vta_avg_1m': 'precio_vta_avg_6m'})
                       )

# Get number of visits by customer, month and different time windows
df_transac_visit = (df_transac[(df_transac.precio_vta_perc.notnull()) &
                              (df_transac.precio_vta_perc > 0)]
                    .reset_index(drop=True)
                    .copy()
                    )
columns_to_keep = ['ID_ficticio', 'year_month', 'fechacompra']
df_transac_visit = df_transac_visit[columns_to_keep]
df_transac_visit.sort_values(['ID_ficticio', 'year_month'], inplace=True)

# Get visit behavior by customer, month and different time windows
df_transac_visit_1m = (df_transac_visit
                       .groupby(['ID_ficticio', 'year_month'])
                       .nunique()
                       .reset_index()
                       .rename(columns={'fechacompra': 'dates_visits_1m'})
                       )
df_transac_visit_3m = (df_transac_visit_1m
                       .groupby(['ID_ficticio'])
                       .rolling(3, on='year_month', min_periods=1)
                       .sum()
                       .reset_index()
                       .drop(columns=['level_1'])
                       .rename(columns={'dates_visits_1m': 'dates_visits_3m'})
                       )
df_transac_visit_6m = (df_transac_visit_1m
                       .groupby(['ID_ficticio'])
                       .rolling(6, on='year_month', min_periods=1)
                       .sum()
                       .reset_index()
                       .drop(columns=['level_1'])
                       .rename(columns={'precio_vta_avg_1m': 'dates_visits_6m'})
                       )

df_summary = (df_transac_venta_1m
              .merge(df_transac_venta_3m,
                     on=['ID_ficticio', 'year_month'],
                     how='left')
              .merge(df_transac_venta_6m,
                     on=['ID_ficticio', 'year_month'],
                     how='left')
              .merge(df_transac_visit_1m,
                     on=['ID_ficticio', 'year_month'],
                     how='left')
              .merge(df_transac_visit_3m,
                     on=['ID_ficticio', 'year_month'],
                     how='left')
              .merge(df_transac_visit_6m,
                     on=['ID_ficticio', 'year_month'],
                     how='left')
    )

df_summary.to_csv('summary_transacciones_clientes.csv', index=False)